In [1]:
import pandas as pd

df = pd.read_csv('./data/logistics_dirty_dataset.csv')

In [2]:
""" Drop duplicate rows from data frame"""
df = df.drop_duplicates()
df.shape

(150000, 20)

## Dropping the Notes and Remarks columns

In [3]:
""" I am dropping the Notes and Remarks columns bcos they don't really have a major impact on the data"""
df = df.drop(labels=['Notes', 'Remarks'], axis=1)


## Fixing the nulls in the Shipment_ID

In [4]:
""" Fixing the nulls in the Shipment_ID column by generating new set of ID for nulls"""
# replace_IDs = [f'SHPFILL{num}' for num in range(1, 1475, 1)]

null_count = df['Shipment_ID'].isnull().sum()
replace_IDs = [f'SHPFILL{num}' for num in range(1, null_count + 1)]
mask = df['Shipment_ID'].isnull()
df.loc[mask, 'Shipment_ID'] = replace_IDs

df['Shipment_ID'].value_counts()

Shipment_ID
SHP58805      10
SHP59414       9
SHP36775       9
SHP53133       9
SHP49235       9
              ..
SHP15658       1
SHPFILL696     1
SHP76989       1
SHP64802       1
SHP33043       1
Name: count, Length: 74038, dtype: int64

## Standardizing all categorical columns

In [5]:
""" Standardizing all categorical columns, fixing whitespace, upper/lower case non-uniformity and typo error"""

category_columns = ['Payment_Method', 'Insurance', 'Destination', 'Origin_City', 'Priority', 'Vehicle_Type', 'Status']
for column in category_columns:
    df[f'{column}'] = df[f'{column}'].str.strip().str.title()
    print(df[f'{column}'].unique())



df['Payment_Method'] = df['Payment_Method'].replace({'Tranfer': 'Transfer'})
df['Vehicle_Type'] = df['Vehicle_Type'].replace({'Bke': 'Bike'})
df['Status'] = df['Status'].replace({'Delayd': 'Delayed'})

print()
print(f'Payment_Method: {df['Payment_Method'].unique()} \n Vehicle_Type: {df['Vehicle_Type'].unique()} \n Status: {df['Status'].unique()} ')

['Cash' 'Card' 'Transfer' 'Tranfer']
['No' 'Yes']
['Customer Hub' 'Retail Store']
['Abuja' 'Lagos' 'Port Harcourt' 'Kano']
['High' 'Medium' 'Low']
['Van' 'Bke' 'Truck' 'Bike']
['Delivered' 'In Transit' 'Cancelled' 'Delayed' 'Delayd']

Payment_Method: ['Cash' 'Card' 'Transfer'] 
 Vehicle_Type: ['Van' 'Bike' 'Truck'] 
 Status: ['Delivered' 'In Transit' 'Cancelled' 'Delayed'] 


## Fixing the date columns

In [6]:
""" Fixing the date columns"""
df['Shipment_Date'] = pd.to_datetime(df['Shipment_Date'], errors='coerce')
df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'], errors='coerce')

print(f'Shipment_Date has: {df['Shipment_Date'].isnull().sum()} null values \n Delivery_Date has: {df['Delivery_Date'].isnull().sum()} null values')

""" I decided to drop the NaT values because a logistics dataset is meant to have a valid Shipment_Date and Delivery_Date"""

df = df.dropna(subset=['Shipment_Date', 'Delivery_Date'])

print(f'Rows remaining after dropping invalid dates: {df.shape[0]:,}')

Shipment_Date has: 2964 null values 
 Delivery_Date has: 3059 null values
Rows remaining after dropping invalid dates: 144,036


C:\Users\Admin\AppData\Local\Temp\ipykernel_12672\113057267.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'], errors='coerce')


## Handling Negative values

In [7]:
""" Handling Negative values """
""" I convert the negative values to NaN first"""
""" I then filled up those NaN with the median of each column"""

cols_negative = ['Distance_km', 'Delivery_Days', 'Cost_NGN', 'Weight_kg']
for column in cols_negative:
    df.loc[df[column] < 0, column] = pd.NA
    median_val = df[column].median()
    df[column] = df[column].fillna(median_val)


## More Fixes

In [8]:
""" Fixing the data type of Delivery_Days as wellas the Nulls + datatype of Customer_Rating"""

df['Delivery_Days'] = df['Delivery_Days'].astype(int)

median_rating = df[ 'Customer_Rating'].median()
df['Customer_Rating'] = df['Customer_Rating'].fillna(median_rating)
df['Customer_Rating'] = df['Customer_Rating'].astype(int)


## Delivery_Duration mismatch check

In [9]:
""" Here i created a new column more like a validation column 'Delivery_Duration' 
which calculates the actual number of days between shipment and delivery"""

df['Delivery_Duration'] = (df['Delivery_Date'] - df['Shipment_Date']).dt.days
df['Delivery_Duration'] = df['Delivery_Duration'].astype(int)
mismatch_check = (df['Delivery_Duration'] != df['Delivery_Days']).sum()

print(f'Rows where Delivery_Duration is not same as Delivery_Days: {mismatch_check:,}')



Rows where Delivery_Duration is not same as Delivery_Days: 2,802


## `Feature Engineering`

### Creating new columns

In [10]:
""" Month, Year, Quater and DayOfWeek columns from Shipment_Date """
df['Shipment_Month'] = df['Shipment_Date'].dt.strftime('%B')
df['Shipment_Year'] = df['Shipment_Date'].dt.strftime('%Y')
df['Shipment_Quater'] = df['Shipment_Date'].dt.quarter.map({1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'})
df['Shipment_DayOfWeek'] = df['Shipment_Date'].dt.strftime('%A')


""" Cost efficiency of each shipment """
df['Cost_per_km'] = (df['Cost_NGN'] / df['Distance_km']).round(2)
""" Cost per unit weight """
df['Cost_per_kg'] = (df['Cost_NGN'] / df['Weight_kg']).round(2)


""" A boolean column flagging whether a shipment was late or not
Is_Late is defined as Status == 'Delayed'. Shipments with Status 'In Transit' are excluded as their final outcome is unknown
"""
df['Is_Late'] = (df['Status'] == 'Delayed')


""" Grouping distance_km into 3 categories"""
distance_bin = [0, 300, 600, float('inf')]
distance_bin_labels = ['Short', 'Medium', 'Long']
df['Distance_Band'] = pd.cut(df['Distance_km'], bins=distance_bin, labels=distance_bin_labels, include_lowest=True)

""" Grouping weight_kg into 3 categories"""
weight_bin = [0, 10.0, 20.0, float('inf')]
weight_bin_labels = ['Light', 'Medium', 'Heavy']
df['Weight_Band'] = pd.cut(df['Weight_kg'], bins=weight_bin, labels=weight_bin_labels, include_lowest=True)

# print(df.shape)
# print()
# print(df[['Cost_per_km', 'Cost_per_kg', 'Is_Late', 'Distance_Band', 'Weight_Band']].head(10))
# print()
# print(f'{df['Distance_Band'].value_counts()}  \n \n  {df['Weight_Band'].value_counts()}')

## FINAL VALIDATION

In [11]:
""" FINAL VALIDATION """

df.duplicated().sum()
df.isnull().sum()

rows, cols = df.shape
print(f'A total of {(150500 - rows):,} rows were lost from the original 150,500')

category_cols = ['Payment_Method', 'Insurance', 'Destination', 'Origin_City', 'Priority', 'Vehicle_Type', 'Status']
for column in category_cols:
    print(df[f'{column}'].unique())

print(f'Remaining duplicates: {df.duplicated().sum()}')
print()
print('Remaining nulls:')
print(df.isnull().sum())


A total of 6,464 rows were lost from the original 150,500
['Cash' 'Card' 'Transfer']
['No' 'Yes']
['Customer Hub' 'Retail Store']
['Abuja' 'Lagos' 'Port Harcourt' 'Kano']
['High' 'Medium' 'Low']
['Van' 'Bike' 'Truck']
['Delivered' 'In Transit' 'Cancelled' 'Delayed']
Remaining duplicates: 0

Remaining nulls:
Shipment_ID           0
Order_ID              0
Origin_City           0
Warehouse             0
Destination           0
Weight_kg             0
Cost_NGN              0
Distance_km           0
Delivery_Days         0
Shipment_Date         0
Delivery_Date         0
Status                0
Vehicle_Type          0
Payment_Method        0
Priority              0
Insurance             0
Driver_Name           0
Customer_Rating       0
Delivery_Duration     0
Shipment_Month        0
Shipment_Year         0
Shipment_Quater       0
Shipment_DayOfWeek    0
Cost_per_km           0
Cost_per_kg           0
Is_Late               0
Distance_Band         0
Weight_Band           0
dtype: int64


## Exporting the Cleaned Dataset

In [12]:
df.to_csv('./data/logistics_CLEANED_data.csv', index=False)
print(f'Clean dataset exported: {df.shape[0]:,} rows, {df.shape[1]} columns')

Clean dataset exported: 144,036 rows, 28 columns
